In [7]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [8]:
import json
import random
from utils import distancia_manhattan_km, calcular_entropia_total

In [9]:
# --- Parametros ---
L = 20 # Capacidad máxima camion
BICIS_INICIALES = 7 # Bicis iniciales en el camión
OBJETIVO = 0.50 # Proporción objetivo de llenado
ESTACION_INICIO = 0 # Estación de inicio
TOLERANCIA = 1 # Margen para considerar una estación "optima" y no visitarla

# --- Datos de los Escenarios (Casos) ---
CASOS = {
    "Caso 1": {
        "bicis": [5, 7, 13, 6, 8, 13, 8, 9, 6, 10, 10, 18, 8, 13, 15, 14],
        "capacidad": [16, 16, 23, 13, 21, 17, 14, 13, 17, 16, 27, 18, 12, 18, 18, 19]
    },
    "Caso 2": {
        "bicis": [15, 14, 3, 10, 2, 1, 8, 13, 17, 1, 20, 9, 6, 6, 15, 14],
        "capacidad": [16, 16, 23, 13, 21, 17, 14, 13, 17, 16, 27, 18, 12, 18, 18, 19]
    },
    "Caso 3": {
        "bicis": [15, 14, 3, 10, 2, 1, 8, 13, 10, 0, 0, 0, 0, 0, 0, 0],
        "capacidad": [15, 14, 3, 10, 2, 1, 8, 13, 17, 16, 27, 18, 12, 18, 18, 19]
    }
}

# --- Seeds para reproducibilidad ---
SEMILLAS = [42, 123, 987, 555, 2024]

# --- Cargar Coordenadas de las estaciones ---
def cargar_coordenadas(ruta_archivo):
    with open(ruta_archivo, 'r') as f:
        return json.load(f)

coordenadas = cargar_coordenadas('coords.json')

NUM_ESTACIONES = len(coordenadas) # Número total de estaciones

In [10]:
def evaluar_ruta(ruta, caso_bicis, caso_capacidad, coordenadas):
    """
    Simula el recorrido del camión y evalúa la ruta dada.

    Args:
        ruta (list): Lista de IDs de estaciones en el orden de visita.
        caso_bicis (dict): Lista con el estado inicial de bicis en cada estación.
        caso_capacidad (dict): Lista con la capacidad máxima de cada estación.
        coordenadas (dict): Diccionario con las coordenadas de cada estación.
    Returns:
        Diccionario con los kms recorridos,
        Entropia Final,
        Estado final de las bicis.
    """
    
    # Copia de los datos para no modificar los originales
    bicis_actuales = list(caso_bicis)
    
    # Variables de estado iniciales
    distancia_total = 0.0
    camion_bicis = BICIS_INICIALES
    capacidad_camion = L
    estacion_actual = ESTACION_INICIO
    
    # Función auxiliar para procesar el intercambio en una estación
    def procesar_intercambio(id):
        nonlocal camion_bicis # Para modificar la var externa
        # Calcolo de objetivo (50% de la capacidad)
        objetivo = int(caso_capacidad[id] * 0.5)
        
        if bicis_actuales[id] > objetivo:
            # La estación tiene exceso -> El camión recoge bicis
            a_recoger = bicis_actuales[id] - objetivo
            espacio_camion = capacidad_camion - camion_bicis
            # Recoge lo que necesita, o hasta llenar el camión
            recogidas = min(a_recoger, espacio_camion)
            bicis_actuales[id] -= recogidas
            camion_bicis += recogidas
            
        elif bicis_actuales[id] < objetivo:
            # La estación tiene déficit -> El camión deja bicis
            a_dejar = objetivo - bicis_actuales[id]
            # Deja lo que necesita, o hasta vaciar el camión
            dejadas = min(a_dejar, camion_bicis)
            bicis_actuales[id] += dejadas
            camion_bicis -= dejadas
            
    # Procesar la Estación 0 antes de salir a la ruta
    procesar_intercambio(ESTACION_INICIO)
    
    # Recorrer la ruta dada
    for proxima_estacion in ruta:
        # Extraer coordenadas
        lat1, lon1 = coordenadas[estacion_actual]['lat'], coordenadas[estacion_actual]['lon']
        lat2, lon2 = coordenadas[proxima_estacion]['lat'], coordenadas[proxima_estacion]['lon']
        
        # Sumar distancia
        distancia_total += distancia_manhattan_km(lat1, lon1, lat2, lon2)
        
        # Actualizar posición del camión
        estacion_actual = proxima_estacion
        
        # Procesar intercambio en la nueva estación
        procesar_intercambio(estacion_actual)
        
    # Volver a la Estación 0 al final de la ruta
    lat1, lon1 = coordenadas[estacion_actual]['lat'], coordenadas[estacion_actual]['lon']
    lat2, lon2 = coordenadas[ESTACION_INICIO]['lat'], coordenadas[ESTACION_INICIO]['lon']
    distancia_total += distancia_manhattan_km(lat1, lon1, lat2, lon2)
    
    # Calcular entropía final
    entropia_final = calcular_entropia_total(bicis_actuales, caso_capacidad)
    
    # Retornar resultados
    return {
        'kms_recorridos': distancia_total,
        'entropia': entropia_final,
        'bicis_finales': bicis_actuales
    }

In [11]:
def obtener_estaciones_a_visitar(caso_bicis, caso_capacidad, tolerancia=TOLERANCIA):
    """
    Filtra las estaciones que necesitan ser visitadas según la tolerancia.
    Excluye la Estación 0, ya que es el origen/destino fijo.

    Args:
        caso_bicis (list): Lista con el estado inicial de bicis en cada estación.
        caso_capacidad (list): Lista con la capacidad máxima de cada estación.
        tolerancia (float): Margen para considerar una estación "optima" y no visitarla.
    Returns:
        Listado de IDs de estaciones a visitar.
    """
    estaciones_validas = []
    
    for i in range(1, len(caso_bicis)):
        # Calculo del 50% ideal
        objetivo = int(caso_capacidad[i] * 0.5)
        
        # Calculo cuantas bicis faltan o sobran
        diferencia = abs(caso_bicis[i] - objetivo)
        
        # Si la diferencia es mayor a la tolerancia, se considera para visitar
        if diferencia > tolerancia:
            estaciones_validas.append(i)
            
    return estaciones_validas

def generar_solucion_inicial_aleatoria(estaciones_base):
    """
    Genera una solución inicial aleatoria a partir de las estaciones a visitar.

    Args:
        estaciones_base (list): Listado de IDs de estaciones a visitar.   
    Returns:
        Lista de IDs de estaciones en un orden random.
    """
    solucion = list(estaciones_base) # Copia de la lista base
    random.shuffle(solucion) # Mezcla aleatoria
    return solucion

In [13]:
from algorithms import greedy_algorithm

for nombre_caso, datos_caso in CASOS.items():
    print(f"\n--- {nombre_caso} ---")
    bicis = datos_caso['bicis']
    capacidad = datos_caso['capacidad']
    
    # Enfoque a priori: Filtrar estaciones a visitar
    estaciones_a_visitar = obtener_estaciones_a_visitar(bicis, capacidad, TOLERANCIA)
    
    # Costruir la ruta Greedy
    ruta_greedy = greedy_algorithm(estaciones_a_visitar, coordenadas)
    
    # Evaluar la ruta Greedy
    resultado_greedy = evaluar_ruta(ruta_greedy, bicis, capacidad, coordenadas)
    
    # Imprimir resultados
    print(f"Ruta Greedy: {ruta_greedy}")
    print(f"Kms Recorridos: {resultado_greedy['kms_recorridos']:.2f}")
    print(f"Entropía Final: {resultado_greedy['entropia']:.4f}")


--- Caso 1 ---
Ruta Greedy: [10, 9, 4, 5, 2, 7, 8, 13, 12, 11, 15, 14]
Kms Recorridos: 23.68
Entropía Final: 15.2903

--- Caso 2 ---
Ruta Greedy: [10, 9, 4, 5, 2, 3, 1, 7, 8, 13, 15, 14]
Kms Recorridos: 23.77
Entropía Final: 14.7736

--- Caso 3 ---
Ruta Greedy: [10, 9, 1, 7, 8, 3, 2, 6, 13, 12, 11, 15, 14]
Kms Recorridos: 23.24
Entropía Final: 9.3560
